In [39]:
from __future__ import annotations
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

## Constellation parameters 16-QAM

In [40]:
SQRT10 = np.sqrt(10.0)

LEVELS = np.array([-3.0 / SQRT10, -1.0 / SQRT10, 1.0 / SQRT10, 3.0 / SQRT10])

AXIS_BITS = {
    LEVELS[2]: "00",
    LEVELS[3]: "01",
    LEVELS[1]: "10",
    LEVELS[0]: "11",
}

## Loading data from csv

In [41]:
def load_channel_data(csv_path):
    df = pd.read_csv(
        csv_path,
        header=None,
        names = ["tx_I", "tx_Q", "rx_I", "rx_Q"],
    )

    tx = df["tx_I"] + df["tx_Q"] * 1j
    rx = df["rx_I"] + df["rx_Q"] * 1j
    return np.array(tx), np.array(rx)


## Building a constellation

In [42]:
def bit_constellation():
    points = []
    labels = []

    for i_level, j_level in product(LEVELS, repeat=2):
        points.append(i_level + j_level * 1j)
        labels.append(AXIS_BITS[i_level] + AXIS_BITS[j_level])
    return np.array(points), np.array(labels)


In [43]:
def demodulate(receive_points, constellation):
    distances = np.abs(receive_points[:, None] - constellation[None, :])
    idx = np.argmin(distances, axis=1)
    return idx

In [44]:
def indexes_to_bit_array(indexes, bit_labels):
    bit_table = np.array([list(map(int, point)) for point in bit_labels])
    bit_sequence = bit_table[indexes]
    return bit_sequence.reshape(-1)


In [45]:
def calculate_ber(rx_bits, tx_bits):
    return sum(rx_bits ^ tx_bits) / rx_bits.size

In [46]:
csv_path  = "Symbols_1m_1ch_PR_1.csv"
tx, rx = load_channel_data(csv_path)
constellation, bit_labels = bit_constellation()


In [50]:
rx_idx = demodulate(rx, constellation)
tx_idx = demodulate(tx, constellation)

rx_bits = indexes_to_bit_array(rx_idx, bit_labels)
tx_bits = indexes_to_bit_array(tx_idx, bit_labels)

ber = calculate_ber(rx_bits, tx_bits)

print(f"BER = {ber:.6f} ({ber * 100:.4f}%)")

BER = 0.008102 (0.8102%)
